# Feature Engineering

______
## 1. Chuẩn bị vấn đề

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import *
import pandas as pd

In [2]:
spark = SparkSession.builder.getOrCreate()

In [3]:
def read_csv(path):
    df = spark.read.csv(path, header=True)
    return df

customers = read_csv("../data/2_clean/customers.csv")
orders = read_csv("../data/2_clean/orders.csv")
order_items = read_csv("../data/2_clean/order_items.csv")
payments = read_csv("../data/2_clean/payments.csv")
products = read_csv("../data/2_clean/products.csv")
reviews = read_csv("../data/2_clean/reviews.csv")

____
## 2. Featuring Data

### 2.1 Chỉ số chính của khách hàng và đơn hàng

```cmd
Tính Customer360 metrics
    
1. Vòng đời khách hàng: first_purchase, last_purchase, customer_age_days, customer_lifetime_days
    - tính trên tất cả orders
2. Đơn hàng: total_orders, delivered_orders, avg_num_of_payments
    - total_orders: tất cả orders
    - delivered_orders, avg_num_of_payments: tính trên delivered orders
3. Giao hàng: num_of_late_delivery, num_of_on_time_delivery, late_rate, avg_delivery_days
    - chỉ delivered orders
4. Địa lý: customer_city, customer_state
    - lấy first
5. RFM: recency_days, monetary
    - chỉ delivered orders
```

In [4]:
def calculate_metrics_customer_order(df, last_date):
    # Tách df delivered
    df_delivered = df.filter(col("order_status") == "delivered")

    # Vòng đời & total_orders (tất cả orders)
    df_all = df.groupBy("customer_unique_id").agg(
        min(col("order_purchase_timestamp")).alias("first_purchase"),
        max(col("order_purchase_timestamp")).alias("last_purchase"),
        count_distinct("order_id").alias("total_orders"),
        first("customer_city").alias("customer_city"),
        first("customer_state").alias("customer_state")
    )

    # Metrics delivered: delivery + monetary + avg_num_of_payments
    df_delivered_agg = df_delivered.groupBy("customer_unique_id").agg(
        sum(when(col("order_estimated_delivery_date") < col("order_delivered_customer_date"), 1).otherwise(0)).alias("num_of_late_delivery"),
        sum(when(col("order_estimated_delivery_date") >= col("order_delivered_customer_date"), 1).otherwise(0)).alias("num_of_on_time_delivery"),
        round(avg(date_diff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))), 2).alias("avg_delivery_days"),
        round(sum("total_value"), 2).alias("monetary"),
        avg("num_of_payments").alias("avg_num_of_payments"),
        count_distinct("order_id").alias("delivered_orders")
    )

    # Join vòng đời + delivered metrics
    df = df_all.join(df_delivered_agg, on="customer_unique_id", how="left")

    # Tính các cột bổ trợ
    df = df.withColumn("customer_age_days", date_diff(lit(last_date), col("first_purchase"))) \
           .withColumn("customer_lifetime_days", date_diff(col("last_purchase"), col("first_purchase")) + 1) \
           .withColumn("late_rate", round(col("num_of_late_delivery") / 
                                          (col("num_of_late_delivery") + col("num_of_on_time_delivery")) * 100, 2)) \
           .withColumn("recency_days", date_diff(lit(last_date), col("last_purchase"))) \
           .withColumn("avg_order_value", round(col("monetary") / (col("delivered_orders")), 2)) \
           .withColumn("orders_per_month", when(col("customer_age_days") > 30, round(col("total_orders") / ((col("customer_age_days") / 30)), 2)).otherwise(1))
           

    return df

### 2.2 Chỉ số khách hàng và phương Thức Thanh Toán (Customer-Payment)

- Các phương thức thanh toán người dùng sử dùng (preferred_payment_type)
- Tổng tiền khách đã chi (total_value)
- Số lần góp tiền trung bình của khách (num_of_payments)

In [5]:
def calculate_metrics_payment(df):
    '''
    Payment trên mỗi order: 
        - total_value          : Tổng tiền khách trả cho 1 đơn
        - preferred_payment_type : Loại payment hay dùng nhất trong order
    '''
    df = df.groupBy("order_id").agg(
        round(sum("payment_value"), 2).alias("total_value"),
        count("order_id").alias("num_of_payments"),
    )
    return df

In [6]:
def calculate_preferred_payment_type(customers, orders, payments):
    df = customers.join(orders, on="customer_id", how="left") \
                  .join(payments, on="order_id", how="left")
    return df.groupBy("customer_unique_id").agg(concat_ws("/", collect_set("payment_type")).alias("preferred_payment_type"))

### 2.3 Chỉ số khách hàng và đánh giá đơn hàng (Customer-Review)

- Tính điểm đánh giá trung bình của khách (avg_review_score)

In [7]:
def calculate_avg_review_score(customers, orders, reviews):
    df = customers.join(orders, on="customer_id", how="left") \
                  .join(reviews, on="order_id", how="left")
    return df.groupBy("customer_unique_id").agg(round(avg(col("review_score")),2).alias("avg_review_score"))

### 2.4 Chỉ số khách hàng và sản phẩm (Customer-Product)

- Top 3 loại sản phẩm được mua theo từng người dùng (top_3_categories)

In [8]:
def calculate_top_3_categories(customers, orders, order_items, products):
    df = customers.join(orders, on="customer_id", how="left") \
                  .join(order_items, on="order_id", how="left") \
                  .join(products, on="product_id", how="left") 
    return df.groupBy("customer_unique_id").agg(concat_ws("/", collect_set("product_category_name")).alias("top_3_categories"))

### 2.5 Tính điểm cho mô hình RFM 

In [9]:
def calculate_RFM_scores(df):
    # Recency
    windowR = Window.orderBy(col("recency_days").asc())
    df = df.withColumn("recency_percentile", percent_rank().over(windowR))
    df = df.withColumn("r_score", 
                       when(col("recency_percentile") <= 0.2, 1)
                       .when(col("recency_percentile") <= 0.4, 2)
                       .when(col("recency_percentile") <= 0.6, 3)
                       .when(col("recency_percentile") <= 0.8, 4)
                       .otherwise(5))

    # Frequency
    windowF = Window.orderBy(col("delivered_orders").desc())
    df = df.withColumn("frequency_percentile", percent_rank().over(windowF))
    df = df.withColumn("f_score", 
                       when(col("frequency_percentile") <= 0.2, 1)
                       .when(col("frequency_percentile") <= 0.4, 2)
                       .when(col("frequency_percentile") <= 0.6, 3)
                       .when(col("frequency_percentile") <= 0.8, 4)
                       .otherwise(5))

    # Monetary
    windowM = Window.orderBy(col("monetary").desc())
    df = df.withColumn("monetary_percentile", percent_rank().over(windowM))
    df = df.withColumn("m_score", 
                       when(col("monetary_percentile") <= 0.2, 1)
                       .when(col("monetary_percentile") <= 0.4, 2)
                       .when(col("monetary_percentile") <= 0.6, 3)
                       .when(col("monetary_percentile") <= 0.8, 4)
                       .otherwise(5))

    df = df.withColumn("rfm_score", concat(col("r_score"), col("f_score"), col("m_score")))
    return df

### 2.6 Tổng hợp và xử lý dữ liệu

#### Xử lý giá trị null

In [10]:
def fill_null_values(df):
    return df.fillna({
        "delivered_orders": 0,
        "monetary": 0,
        "avg_order_value": 0,
        "avg_num_of_payments": 0,
        "num_of_on_time_delivery": 0,
        "num_of_late_delivery": 0,
        "late_rate": 0,
        "avg_delivery_days": 0,
    })

#### Sắp xếp cột

In [11]:
def select_customer360(df):
    return df.select(
        "customer_unique_id",
        "first_purchase", "last_purchase", "customer_age_days", "customer_lifetime_days", "recency_days",
        "total_orders", "delivered_orders", "monetary", "avg_order_value", "orders_per_month",
        "avg_num_of_payments", "preferred_payment_type", "avg_review_score", "top_3_categories",
        "r_score", "f_score", "m_score", "rfm_score",
        "num_of_on_time_delivery", "num_of_late_delivery", "late_rate", "avg_delivery_days",
        "customer_city", "customer_state"
    )

#### Main

In [12]:
def generate_customer360_df(customers, orders, order_items, payments, products, reviews, last_date):
    preferred_payment_type_df = calculate_preferred_payment_type(customers, orders, payments)
    avg_review_score_df = calculate_avg_review_score(customers, orders, reviews)
    top_3_categories_df = calculate_top_3_categories(customers, orders, order_items, products)

    enriched_payments = calculate_metrics_payment(payments)
    orders = orders.join(enriched_payments, on = "order_id", how="left")

    df = customers.join(orders, on = "customer_id", how="left")
    df = calculate_metrics_customer_order(df, last_date)
    df = df.join(preferred_payment_type_df, on = "customer_unique_id", how="left")
    df = df.join(avg_review_score_df, on = "customer_unique_id", how="left")
    df = df.join(top_3_categories_df, on = "customer_unique_id", how="left")

    df = calculate_RFM_scores(df)
    
    df = fill_null_values(df)
    df = select_customer360(df)

    return df


_____
## 3. Kết quả

In [13]:
def write_to_disk(df):
    pandas_df = df.toPandas()
    pandas_df.to_csv("../data/3_model/featured_data.csv", index=False)

In [16]:
if __name__ == "__main__":
    last_date = orders.agg(max("order_purchase_timestamp")).collect()[0][0]
    customer360_df = generate_customer360_df(customers, orders, order_items, payments, products, reviews, last_date)
    write_to_disk(customer360_df)
    customer360_df.show()
    print(customer360_df.count())

+--------------------+-------------------+-------------------+-----------------+----------------------+------------+------------+----------------+--------+---------------+----------------+-------------------+----------------------+----------------+--------------------+-------+-------+-------+---------+-----------------------+--------------------+---------+-----------------+------------------+--------------+
|  customer_unique_id|     first_purchase|      last_purchase|customer_age_days|customer_lifetime_days|recency_days|total_orders|delivered_orders|monetary|avg_order_value|orders_per_month|avg_num_of_payments|preferred_payment_type|avg_review_score|    top_3_categories|r_score|f_score|m_score|rfm_score|num_of_on_time_delivery|num_of_late_delivery|late_rate|avg_delivery_days|     customer_city|customer_state|
+--------------------+-------------------+-------------------+-----------------+----------------------+------------+------------+----------------+--------+---------------+-------

In [1]:
import pandas as pd

test = pd.read_csv("../data/3_model/featured_data.csv")
test

,customer_unique_id,first_purchase,last_purchase,customer_age_days,customer_lifetime_days,recency_days,total_orders,delivered_orders,monetary,avg_order_value,...,r_score,f_score,m_score,rfm_score,num_of_on_time_delivery,num_of_late_delivery,late_rate,avg_delivery_days,customer_city,customer_state
0,0a0a92112bd4c708ca5fde585afaa872,2017-09-29 15:24:52,2017-09-29 15:24:52,383,1,383,1,1,13664.08,13664.08,...,4,1,1,411,1,0,0.0,18.0,rio de janeiro,RJ
1,da122df9eeddfedc1dc1f5349a1a690c,2017-04-01 15:58:40,2017-04-01 15:58:41,564,1,564,2,2,7571.63,3785.82,...,5,1,1,511,2,0,0.0,16.0,araruama,RJ
2,763c8b1c9c68a0229c42c9fc6f662b93,2018-07-15 14:49:44,2018-07-15 14:49:44,94,1,94,1,1,7274.88,7274.88,...,1,1,1,111,1,0,0.0,11.0,vila velha,ES
3,dc4802a71eae9be1dd28f5d788ceb526,2017-02-12 20:37:36,2017-02-12 20:37:36,612,1,612,1,1,6929.31,6929.31,...,5,1,1,511,1,0,0.0,19.0,campo grande,MS
4,459bef486812aa25204be022145caa62,2018-07-25 18:10:17,2018-07-25 18:10:17,84,1,84,1,1,6922.21,6922.21,...,1,1,1,111,0,1,100.0,21.0,vitoria,ES
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96091,b8b8726af116a5cfb35b0315ecef9172,2016-10-03 21:01:41,2016-10-03 21:01:41,744,1,744,1,0,0.00,0.00,...,5,5,5,555,0,0,0.0,0.0,rio de janeiro,RJ
96092,0eb1ee9dba87f5b36b4613a65074337c,2016-10-02 22:07:52,2016-10-02 22:07:52,745,1,745,1,0,0.00,0.00,...,5,5,5,555,0,0,0.0,0.0,sao paulo,SP
96093,009b0127b727ab0ba422f6d9604487c7,2016-09-13 15:24:19,2016-09-13 15:24:19,764,1,764,1,0,0.00,0.00,...,5,5,5,555,0,0,0.0,0.0,sao jose dos campos,SP
96094,4854e9b3feff728c13ee5fc7d1547e92,2016-09-05 00:15:34,2016-09-05 00:15:34,772,1,772,1,0,0.00,0.00,...,5,5,5,555,0,0,0.0,0.0,passo fundo,RS


In [2]:
test[test["total_orders"] > 1]

,customer_unique_id,first_purchase,last_purchase,customer_age_days,customer_lifetime_days,recency_days,total_orders,delivered_orders,monetary,avg_order_value,...,r_score,f_score,m_score,rfm_score,num_of_on_time_delivery,num_of_late_delivery,late_rate,avg_delivery_days,customer_city,customer_state
1,da122df9eeddfedc1dc1f5349a1a690c,2017-04-01 15:58:40,2017-04-01 15:58:41,564,1,564,2,2,7571.63,3785.82,...,5,1,1,511,2,0,0.0,16.0,araruama,RJ
9,c8460e4251689ba205045f3ea17884a1,2018-08-07 09:03:02,2018-08-08 14:27:15,71,2,70,4,4,4655.91,1163.98,...,1,1,1,111,4,0,0.0,12.0,porto alegre,RS
24,59d66d72939bc9497e19d89c61a96d5f,2017-03-02 12:13:18,2017-08-10 22:09:50,594,162,433,2,2,3559.99,1780.00,...,4,1,1,411,1,1,50.0,16.5,sao paulo,SP
35,46450c74a0d8c5ca9395da1daac6c120,2018-07-24 20:41:01,2018-08-17 20:06:36,85,25,61,3,1,3184.34,3184.34,...,1,1,1,111,1,0,0.0,9.0,florianopolis,SC
61,eae0a83d752b1dd32697e0e7b4221656,2018-02-01 18:32:02,2018-04-24 17:06:54,258,83,176,2,2,2783.01,1391.51,...,2,1,1,211,2,0,0.0,33.0,cicero dantas,BA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95017,5ace05247b6926d3e595ac4de6620b1d,2017-09-29 22:44:21,2017-09-29 22:44:22,383,1,383,2,0,0.00,0.00,...,4,5,5,455,0,0,0.0,0.0,rio de janeiro,RJ
95362,9eb7d30b5661dd9e5e2333954d84e02a,2017-07-07 21:17:36,2017-07-07 21:17:36,467,1,467,2,0,0.00,0.00,...,5,5,5,555,0,0,0.0,0.0,timburi,SP
95694,0af334fc660bfc9c75109752cc8271d7,2017-04-16 06:31:07,2017-04-16 06:31:08,549,1,549,2,0,0.00,0.00,...,5,5,5,555,0,0,0.0,0.0,rio de janeiro,RJ
95767,8b1aed2dad15fcad8eb2a5c555e2d26e,2017-03-23 22:59:51,2017-03-23 23:44:58,573,1,573,2,0,0.00,0.00,...,5,5,5,555,0,0,0.0,0.0,sao paulo,SP
